# 📦 Modul 1: Chunking & Embedding
> **RAG-Framework** · Phase 2 · Wissenschaftliche Evaluation

Dieses Notebook evaluiert systematisch verschiedene **Chunking-Strategien** und **Embedding-Modelle**.
Am Ende steht eine messbare Entscheidungsgrundlage für die Produktivarchitektur.

---
### Inhalt
1. Setup & Imports
2. Testdaten laden
3. Chunking-Strategien im Vergleich
4. Embedding-Modell-Evaluation
5. Retrieval-Qualitäts-Benchmark
6. Ergebnisse & Empfehlung


---
## 1. Setup & Imports

In [3]:
# ── Installation (nur beim ersten Ausführen nötig) ─────────────────────────
!pip install langchain langchain-community langchain-openai \
             langchain-experimental langchain-text-splitters \
             sentence-transformers openai tiktoken \
             qdrant-client chromadb scikit-learn \
             matplotlib seaborn pandas numpy \
             python-dotenv tqdm --quiet 

In [ ]:
# Zelle 1 — Pakete reparieren
#!pip install tf-keras --quiet
#!pip install protobuf==3.20.3 --quiet
#!pip install ipywidgets widgetsnbextension --quiet
#!pip install --upgrade jupyter --quiet

#!pip uninstall tensorflow tensorflow-intel tensorflow-cpu -y --quiet
#!pip uninstall transformers -y --quiet
#!pip install transformers==4.44.0 --quiet
#!pip install sentence-transformers==2.7.0 --quiet

You can safely remove it manually.
You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"]  = "TRUE"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["OMP_NUM_THREADS"]        = "1"
os.environ["MKL_NUM_THREADS"]        = "1"
os.environ["NUMEXPR_NUM_THREADS"]    = "1"

import onnxruntime  # muss vor torch importiert werden (Windows DLL-Reihenfolge)
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm.notebook import tqdm
from dotenv import load_dotenv
from IPython.display import display, Markdown

# LangChain Splitters
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter,
)
from langchain_experimental.text_splitter import SemanticChunker

# Embeddings
from langchain_openai import OpenAIEmbeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

# VectorStore (lokal, kein Server nötig)
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

warnings.filterwarnings('ignore')
load_dotenv()  # .env Datei mit OPENAI_API_KEY

# Plotting-Style
plt.style.use('dark_background')
sns.set_palette('husl')
ACCENT = '#00f5d4'

print('✅ Imports erfolgreich')
print(f'   NumPy  {np.__version__}')
print(f'   Pandas {pd.__version__}')

---
## 2. Testdaten laden

> Wir nutzen einen synthetischen Unternehmenstext (Jahresbericht-Stil).
> **Tausche `SAMPLE_TEXT` gegen echte Dokumente aus** (PDF, DOCX, TXT).

In [ ]:
# ── Beispieltext (ersetze durch echte Dokumente) ───────────────────────────
SAMPLE_TEXT = """
Geschäftsbericht 2024 – Mustermann AG

1. Vorwort des Vorstands
Das Geschäftsjahr 2024 war geprägt von signifikantem Wachstum und strategischer Neuausrichtung.
Der Gesamtumsatz stieg um 18,4 Prozent auf 2,3 Milliarden Euro. Besonders die Segmente
Digitalisierung und erneuerbare Energien trugen zu dieser positiven Entwicklung bei.
Die Eigenkapitalrendite lag bei 14,2 Prozent und übertraf damit den Branchendurchschnitt
von 11,8 Prozent deutlich. Für das Geschäftsjahr 2025 erwarten wir ein weiteres Wachstum
von 12 bis 15 Prozent, getragen durch die Expansion in den asiatischen Märkten.

2. Risikoanalyse
Die Risikolandschaft des Unternehmens hat sich im Berichtsjahr verändert. Operative Risiken
wurden durch den Ausbau interner Kontrollsysteme reduziert. Das Klumpenrisiko im Bereich
Lieferkette wurde durch Diversifikation auf 14 Lieferanten minimiert. Währungsrisiken
aus dem Dollargeschäft (ca. 35% des Umsatzes) werden durch Forward-Hedging abgesichert.
Ein neu identifiziertes Cyber-Risiko durch erhöhte Angriffsfläche im Cloud-Betrieb
wird durch ein investiertes Budget von 8,5 Millionen Euro in IT-Security adressiert.
Das Gesamtrisikoprofil wird als beherrschbar eingestuft.

3. Finanzielle Kennzahlen
Der Jahresüberschuss betrug 187 Millionen Euro (Vorjahr: 142 Millionen Euro), was einem
Anstieg von 31,7 Prozent entspricht. Das EBITDA verbesserte sich auf 410 Millionen Euro
mit einer EBITDA-Marge von 17,8 Prozent. Der operative Cashflow war mit 345 Millionen
Euro stark positiv, was die solide Liquiditätssituation des Unternehmens belegt.
Die Nettoverschuldung sank auf 1,2x EBITDA (Vorjahr: 1,8x). Die Dividende wird um
10 Prozent auf 1,65 Euro je Aktie erhöht.

4. Strategie & Ausblick
Die strategischen Säulen des Unternehmens sind: Digitale Transformation, Nachhaltigkeit,
Internationalisierung und Innovationsführerschaft. Im Bereich KI-gestützte Prozesse wurden
bereits 23 Pilotprojekte gestartet, von denen 14 in den Produktivbetrieb überführt wurden.
Die CO2-Emissionen wurden um 22 Prozent reduziert. Für 2025 sind Akquisitionen in
Höhe von bis zu 500 Millionen Euro geplant, vorrangig in den Bereichen Software und
Clean Technology. Der Auftragsbestand per 31.12.2024 beträgt 4,1 Milliarden Euro.

5. Corporate Governance
Der Aufsichtsrat besteht aus 12 Mitgliedern, davon 6 Arbeitnehmervertreter. Im Berichtsjahr
fanden 8 Sitzungen des Aufsichtsrats statt. Die Vergütung des Vorstands ist zu 60 Prozent
an langfristige Nachhaltigkeitsziele geknüpft. Das Compliance-Management-System wurde
nach ISO 37301 zertifiziert. Es gab keine wesentlichen Compliance-Verstöße im Berichtsjahr.
"""

# ── Optional: echtes PDF laden ─────────────────────────────────────────────
# from langchain_community.document_loaders import PyPDFLoader
# loader = PyPDFLoader("data/raw/geschaeftsbericht_2024.pdf")
# pages = loader.load()
# SAMPLE_TEXT = " ".join([p.page_content for p in pages])

print(f'📄 Textlänge: {len(SAMPLE_TEXT):,} Zeichen | {len(SAMPLE_TEXT.split()):,} Wörter')

---
## 3. Chunking-Strategien im Vergleich

| Strategie | Beschreibung | Geeignet für |
|---|---|---|
| **Fixed-Size** | Feste Zeichenanzahl mit Overlap | Einfache Dokumente |
| **Recursive** | Teilt an \n\n → \n → Satz → Wort | Allgemeiner Standard |
| **Token-based** | Chunks nach Token (LLM-nativ) | Präzise Context-Window-Nutzung |
| **Semantic** | Embedding-basierte Satzgrenzen | Hohe Chunk-Kohärenz |

In [ ]:
# ── Chunking-Konfigurationen ───────────────────────────────────────────────
CHUNK_CONFIGS = [
    {"name": "Fixed-256",     "size": 256,  "overlap": 32},
    {"name": "Fixed-512",     "size": 512,  "overlap": 64},
    {"name": "Fixed-1024",    "size": 1024, "overlap": 128},
    {"name": "Recursive-512", "size": 512,  "overlap": 64},
    {"name": "Token-256",     "size": 256,  "overlap": 32},
]

def run_chunking(text: str, configs: list) -> dict:
    results = {}
    
    for cfg in configs:
        name = cfg["name"]
        
        if name.startswith("Fixed"):
            splitter = CharacterTextSplitter(
                chunk_size=cfg["size"],
                chunk_overlap=cfg["overlap"],
                separator="\n"
            )
        elif name.startswith("Recursive"):
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=cfg["size"],
                chunk_overlap=cfg["overlap"],
                separators=["\n\n", "\n", ". ", " ", ""]
            )
        elif name.startswith("Token"):
            splitter = TokenTextSplitter(
                chunk_size=cfg["size"],
                chunk_overlap=cfg["overlap"]
            )
        
        chunks = splitter.split_text(text)
        lengths = [len(c) for c in chunks]
        
        results[name] = {
            "chunks": chunks,
            "count": len(chunks),
            "avg_len": np.mean(lengths),
            "std_len": np.std(lengths),
            "min_len": np.min(lengths),
            "max_len": np.max(lengths),
            "config": cfg
        }
        print(f"  ✓ {name:<20} → {len(chunks):>3} Chunks | Ø {np.mean(lengths):.0f} Zeichen")
    
    return results

print("🔪 Starte Chunking-Evaluation...")
chunking_results = run_chunking(SAMPLE_TEXT, CHUNK_CONFIGS)

In [ ]:
# ── Visualisierung: Chunk-Größenverteilung ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

# Plot 1: Anzahl Chunks
names = list(chunking_results.keys())
counts = [chunking_results[n]['count'] for n in names]
bars = axes[0].bar(names, counts, color=[ACCENT, '#7b2fff', '#ff6b35', '#ffd166', '#06d6a0'], alpha=0.85)
axes[0].set_title('Anzahl Chunks pro Strategie', color='white', pad=12)
axes[0].set_facecolor('#0d1117')
axes[0].tick_params(colors='white', rotation=30)
axes[0].spines[:].set_color('#333')
for bar, val in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                 str(val), ha='center', va='bottom', color='white', fontsize=10)

# Plot 2: Ø Chunk-Länge mit Std-Abweichung
avgs = [chunking_results[n]['avg_len'] for n in names]
stds = [chunking_results[n]['std_len'] for n in names]
colors = [ACCENT, '#7b2fff', '#ff6b35', '#ffd166', '#06d6a0']
axes[1].barh(names, avgs, xerr=stds, color=colors, alpha=0.85, error_kw={'ecolor': 'white', 'capsize': 4})
axes[1].set_title('Ø Chunk-Länge ± Std (Zeichen)', color='white', pad=12)
axes[1].set_facecolor('#0d1117')
axes[1].tick_params(colors='white')
axes[1].spines[:].set_color('#333')

plt.tight_layout()
plt.savefig('chunking_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('💾 Grafik gespeichert: chunking_comparison.png')

In [ ]:
# ── Chunk-Qualitäts-Inspektion ─────────────────────────────────────────────
INSPECT_STRATEGY = "Recursive-512"  # ← hier anpassen
chunks = chunking_results[INSPECT_STRATEGY]['chunks']

print(f"\n🔍 Inspektion: {INSPECT_STRATEGY}")
print(f"   {len(chunks)} Chunks\n")
print("─" * 60)
for i, chunk in enumerate(chunks[:3]):
    print(f"\n[Chunk {i+1}] ({len(chunk)} Zeichen)")
    print(chunk[:300] + "..." if len(chunk) > 300 else chunk)
    print("─" * 60)

---
## 4. Embedding-Modell-Evaluation

Wir testen drei Modell-Klassen:
- **OpenAI** `text-embedding-3-small` — Cloud, kostenpflichtig, sehr stark
- **BGE-M3** (HuggingFace) — lokal, multilingual, SOTA Open-Source
- **all-MiniLM-L6** (HuggingFace) — lokal, schnell, leichtgewichtig


In [ ]:
# ── fastembed installieren (ONNX-basierte Embeddings, stabil auf Windows) ──
!pip install fastembed --quiet

In [ ]:
# ── Embedding-Modelle konfigurieren ───────────────────────────────────────
# fastembed/onnxruntime muss VOR torch importiert werden
from fastembed import TextEmbedding
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Gerät: {device} (torch {torch.__version__})")

USE_OPENAI  = False   # ← True wenn OPENAI_API_KEY gesetzt
SKIP_BGE_M3 = False   # ← True für schnelleren Test (MiniLM reicht meist)

class FastEmbedWrapper:
    """FastEmbed-Wrapper (ONNX Runtime, stabil auf Windows) mit embed_documents/embed_query API."""
    def __init__(self, model_name: str):
        self._model = TextEmbedding(model_name_or_path=model_name)
    def embed_documents(self, texts: list) -> list:
        return [v.tolist() for v in self._model.embed(texts)]
    def embed_query(self, text: str) -> list:
        return next(self._model.query_embed(text)).tolist()

EMBEDDING_MODELS = {
    "MiniLM-L6": FastEmbedWrapper("sentence-transformers/all-MiniLM-L6-v2"),
    **({"BGE-M3": FastEmbedWrapper("BAAI/bge-m3")} if not SKIP_BGE_M3 else {}),
}

if USE_OPENAI:
    from langchain_openai import OpenAIEmbeddings
    EMBEDDING_MODELS["OpenAI-3-small"] = OpenAIEmbeddings(
        model="text-embedding-3-small",
        openai_api_key=os.getenv("OPENAI_API_KEY")
    )

print(f"📐 Embedding-Modelle geladen: {list(EMBEDDING_MODELS.keys())}")

In [ ]:
# ── Embedding-Benchmark: Geschwindigkeit & Dimensionen ────────────────────
TEST_CHUNKS = chunking_results["Recursive-512"]["chunks"][:10]  # 10 Chunks zum Testen

embedding_stats = {}

print("⏱️  Starte Embedding-Benchmark...\n")
for model_name, model in EMBEDDING_MODELS.items():
    start = time.time()
    vectors = model.embed_documents(TEST_CHUNKS)
    elapsed = time.time() - start
    
    vectors_np = np.array(vectors)
    embedding_stats[model_name] = {
        "dims": vectors_np.shape[1],
        "time_sec": elapsed,
        "time_per_chunk": elapsed / len(TEST_CHUNKS),
        "vectors": vectors_np,
        "norm_mean": np.mean(np.linalg.norm(vectors_np, axis=1)),
    }
    print(f"  ✓ {model_name:<20} │ Dims: {vectors_np.shape[1]:>4} │ Zeit: {elapsed:.2f}s │ {elapsed/len(TEST_CHUNKS)*1000:.1f}ms/Chunk")

print("\n✅ Benchmark abgeschlossen")

In [ ]:
# ── Kosinus-Ähnlichkeits-Heatmap ──────────────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity

n_models = len(embedding_stats)
fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 6))
fig.patch.set_facecolor('#0d1117')

if n_models == 1:
    axes = [axes]

for ax, (model_name, stats) in zip(axes, embedding_stats.items()):
    vecs = stats["vectors"]
    sim_matrix = cosine_similarity(vecs)
    
    sns.heatmap(
        sim_matrix, ax=ax,
        cmap='viridis', vmin=0, vmax=1,
        xticklabels=range(1, len(TEST_CHUNKS)+1),
        yticklabels=range(1, len(TEST_CHUNKS)+1),
        annot=True, fmt='.2f', annot_kws={'size': 8}
    )
    ax.set_title(f'Kosinus-Ähnlichkeit\n{model_name}', color='white', pad=10)
    ax.set_facecolor('#0d1117')
    ax.tick_params(colors='white')

plt.tight_layout()
plt.savefig('embedding_similarity.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

display(Markdown("""
**Interpretation der Heatmap:**
- Werte nahe **1.0** = sehr ähnliche Chunks (evtl. zu viel Overlap)
- Werte **0.3–0.7** = gute semantische Distinktheit
- Diagonale immer **1.0** (Chunk mit sich selbst)
"""))

---
## 5. Retrieval-Qualitäts-Benchmark

Wir erstellen einen kleinen Gold-Standard mit **Frage→Erwarteter-Chunk-Paaren** und messen **Precision@K**.

In [ ]:
# ── Gold-Standard: Testfragen für Unternehmensanalyse ─────────────────────
GOLD_QUERIES = [
    {
        "question": "Wie hoch war der Jahresüberschuss 2024?",
        "keywords": ["187 Millionen", "Jahresüberschuss", "31,7 Prozent"]
    },
    {
        "question": "Welche Risiken wurden im Berichtsjahr identifiziert?",
        "keywords": ["Cyber-Risiko", "Klumpenrisiko", "Währungsrisiken", "Lieferkette"]
    },
    {
        "question": "Was sind die strategischen Säulen des Unternehmens?",
        "keywords": ["Digitale Transformation", "Nachhaltigkeit", "Internationalisierung"]
    },
    {
        "question": "Wie hat sich die Dividende verändert?",
        "keywords": ["1,65 Euro", "10 Prozent", "Dividende"]
    },
    {
        "question": "Wie ist der Aufsichtsrat zusammengesetzt?",
        "keywords": ["12 Mitgliedern", "Arbeitnehmervertreter", "Aufsichtsrat"]
    },
]

def keyword_hit_score(chunk_text: str, keywords: list) -> float:
    """Prüft, wie viele Keywords im Chunk vorkommen (0.0–1.0)"""
    hits = sum(1 for kw in keywords if kw.lower() in chunk_text.lower())
    return hits / len(keywords)

print(f'📋 {len(GOLD_QUERIES)} Testfragen geladen')

In [ ]:
# ── Retrieval-Benchmark über alle Chunking × Embedding Kombinationen ──────
# Hinweis: Nutzt direkte numpy Cosinus-Ähnlichkeit statt Chroma (schneller, kein chromadb-Crash)
from sklearn.metrics.pairwise import cosine_similarity as _cos_sim

K_VALUES = [1, 3, 5]
benchmark_rows = []

print("🔬 Starte Retrieval-Benchmark...")
print("   (Strategie × Embedding-Modell × Precision@K)
")

for chunk_name, chunk_data in tqdm(chunking_results.items(), desc="Chunking"):
    chunks = chunk_data["chunks"]

    for model_name, model in EMBEDDING_MODELS.items():
        # Alle Chunk-Embeddings auf einmal berechnen
        chunk_vectors = np.array(model.embed_documents(chunks))

        for k in K_VALUES:
            scores_at_k = []
            for query in GOLD_QUERIES:
                q_vec = np.array(model.embed_query(query["question"])).reshape(1, -1)
                sims = _cos_sim(q_vec, chunk_vectors)[0]
                top_k_idx = np.argsort(sims)[-k:][::-1]
                max_score = max(
                    keyword_hit_score(chunks[i], query["keywords"])
                    for i in top_k_idx
                )
                scores_at_k.append(max_score)

            benchmark_rows.append({
                "Chunking":      chunk_name,
                "Embedding":     model_name,
                "K":             k,
                "Precision@K":   np.mean(scores_at_k),
                "Chunks_Anzahl": chunk_data["count"],
                "Avg_Chunk_Len": round(chunk_data["avg_len"])
            })

        print(f"  ✓ {chunk_name} × {model_name}")

df_results = pd.DataFrame(benchmark_rows)
print("
✅ Benchmark abgeschlossen")
print(df_results.groupby(["Chunking", "Embedding"])[["Precision@K"]].mean().round(3).to_string())

In [ ]:
# ── Ergebnis-Heatmap: Precision@3 ─────────────────────────────────────────
df_p3 = df_results[df_results["K"] == 3].pivot(
    index="Chunking", columns="Embedding", values="Precision@K"
)

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')

sns.heatmap(
    df_p3, ax=ax, cmap='RdYlGn', vmin=0, vmax=1,
    annot=True, fmt='.3f', annot_kws={'size': 12, 'weight': 'bold'},
    linewidths=0.5, linecolor='#1a1a2e',
    cbar_kws={'label': 'Precision@3'}
)
ax.set_title('Retrieval-Qualität: Precision@3\n(Chunking-Strategie × Embedding-Modell)',
             color='white', fontsize=13, pad=14)
ax.tick_params(colors='white')
ax.set_xlabel('Embedding-Modell', color='#aaa')
ax.set_ylabel('Chunking-Strategie', color='#aaa')

plt.tight_layout()
plt.savefig('retrieval_heatmap.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

---
## 6. Ergebnisse & Empfehlung

In [ ]:
# ── Automatische Empfehlung basierend auf Benchmark-Ergebnissen ───────────
best_combo = df_results[df_results["K"] == 3].sort_values("Precision@K", ascending=False).iloc[0]

display(Markdown(f"""
## 🏆 Empfehlung für dein RAG-Framework

| Kriterium | Empfehlung |
|---|---|
| **Beste Kombination** | `{best_combo['Chunking']}` + `{best_combo['Embedding']}` |
| **Precision@3** | `{best_combo['Precision@K']:.3f}` |
| **Chunks** | `{int(best_combo['Chunks_Anzahl'])}` Chunks @ Ø `{int(best_combo['Avg_Chunk_Len'])}` Zeichen |

### Nächste Schritte
1. **Semantic Chunking** testen (erfordert Embedding-Modell schon beim Chunking)
2. Beste Kombination in `src/ingestion/chunker.py` und `src/ingestion/embedder.py` überführen
3. **Qdrant** als Produktiv-VectorStore einbinden (statt Chroma)
4. Weiter zu **Notebook 02**: Advanced RAG mit Re-Ranking & HyDE
"""))

# Ergebnisse speichern
df_results.to_csv('chunking_embedding_benchmark.csv', index=False)
print('💾 Ergebnisse gespeichert: chunking_embedding_benchmark.csv')